In [1]:
# CELL 1: IMPORTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib, time
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, roc_curve, precision_recall_curve
)
import xgboost as xgb

# Optional: LightGBM
try:
    import lightgbm as lgb
    LGBM_OK = True
    print(f' LightGBM {lgb.__version__} available')
except ImportError:
    LGBM_OK = False
    print('  LightGBM not found  →  pip install lightgbm')

os.makedirs('../results', exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})

print('\n' + '='*60)
print('   NOTEBOOK 05: ENSEMBLE METHODS & STACKING')
print('='*60)

  LightGBM not found  →  pip install lightgbm

   NOTEBOOK 05: ENSEMBLE METHODS & STACKING


In [2]:
# CELL 2: LOAD PREPROCESSED DATA + NB03 BASE MODELS

X_train = np.load('../data/X_train.npy')
y_train = np.load('../data/y_train.npy')
X_val   = np.load('../data/X_val.npy')
y_val   = np.load('../data/y_val.npy')
X_test  = np.load('../data/X_test.npy')
y_test  = np.load('../data/y_test.npy')
feature_names = pd.read_csv('../data/feature_names.csv')['feature'].tolist()

# Load the 3 base models saved in NB03
lr_model  = joblib.load('../results/model_lr.pkl')
rf_model  = joblib.load('../results/model_rf.pkl')
xgb_model = joblib.load('../results/model_xgb.pkl')  # or model_xgb_tuned.pkl

print(f'X_train : {X_train.shape}  Fraud: {y_train.mean()*100:.2f}% [SMOTE applied]')
print(f'X_val   : {X_val.shape}    Fraud: {y_val.mean()*100:.3f}%')
print(f'X_test  : {X_test.shape}   Fraud: {y_test.mean()*100:.3f}%')
print(f'\nBase models loaded:')
print(f'  LR  : {lr_model}')
print(f'  RF  : n_estimators={rf_model.n_estimators}')
print(f'  XGB : n_estimators={xgb_model.n_estimators}')

# Base model val-set probabilities (used throughout this notebook)
lr_val_probs  = lr_model.predict_proba(X_val)[:, 1]
rf_val_probs  = rf_model.predict_proba(X_val)[:, 1]
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]

print('\nBase model val-set AUC-ROC:')
for name, probs in [('LR', lr_val_probs),('RF', rf_val_probs),('XGBoost', xgb_val_probs)]:
    print(f'  {name:8s}: {roc_auc_score(y_val, probs):.4f}')

X_train : (200157, 32)  Fraud: 9.09% [SMOTE applied]
X_val   : (45569, 32)    Fraud: 0.173%
X_test  : (56962, 32)   Fraud: 0.172%

Base models loaded:
  LR  : LogisticRegression(C=0.01, class_weight='balanced', max_iter=1000, n_jobs=-1,
                   random_state=42)
  RF  : n_estimators=200
  XGB : n_estimators=500

Base model val-set AUC-ROC:
  LR      : 0.9704
  RF      : 0.9630
  XGBoost : 0.9659


In [3]:
# CELL 3: SOFT VOTING ENSEMBLE
# Why? Averaging probabilities reduces variance.
# Individual model errors partially cancel each other out.
# Even if XGBoost is best, voting occasionally fixes its errors
# using RF or LR predictions.

print('=== Soft Voting Ensembles ===')
print('How it works: average P(fraud) across all base models')
print('  Equal weights  : (LR + RF + XGB) / 3')
print('  Optimal weights: found by grid search on val set')

def eval_probs(probs, y_true, threshold=0.3):
    auc  = roc_auc_score(y_true, probs)
    ap   = average_precision_score(y_true, probs)
    pred = (probs >= threshold).astype(int)
    f1   = f1_score(y_true, pred)
    rec  = recall_score(y_true, pred)
    pre  = precision_score(y_true, pred)
    return {'auc': auc, 'ap': ap, 'f1': f1, 'recall': rec, 'precision': pre}

# Equal voting
vote_equal = (lr_val_probs + rf_val_probs + xgb_val_probs) / 3

# Grid search for best weights (w_lr, w_rf, w_xgb) on validation set
best_auc, best_w = 0, (0.1, 0.2, 0.7)
for w1 in np.arange(0.0, 0.4, 0.05):
    for w2 in np.arange(0.1, 0.5, 0.05):
        w3 = round(1 - w1 - w2, 4)
        if 0 < w3 <= 1:
            blended = w1*lr_val_probs + w2*rf_val_probs + w3*xgb_val_probs
            auc = roc_auc_score(y_val, blended)
            if auc > best_auc:
                best_auc, best_w = auc, (w1, w2, w3)

w_lr, w_rf, w_xgb = best_w
vote_optimal = w_lr*lr_val_probs + w_rf*rf_val_probs + w_xgb*xgb_val_probs

# Report
results_all = {}
for name, probs in [
    ('XGBoost (NB03 base)',  xgb_val_probs),
    ('Equal Voting (1/3 each)', vote_equal),
    (f'Optimal Voting ({w_lr:.2f}/{w_rf:.2f}/{w_xgb:.2f})', vote_optimal),
]:
    m = eval_probs(probs, y_val)
    results_all[name] = {**m, 'probs': probs}
    print(f'\n  {name}')
    print(f'    AUC-ROC={m["auc"]:.4f}  AP={m["ap"]:.4f}  F1={m["f1"]:.4f}  Recall={m["recall"]:.4f}  Precision={m["precision"]:.4f}')

print(f'\n  Optimal weights: LR={w_lr:.2f}, RF={w_rf:.2f}, XGB={w_xgb:.2f}')

=== Soft Voting Ensembles ===
How it works: average P(fraud) across all base models
  Equal weights  : (LR + RF + XGB) / 3
  Optimal weights: found by grid search on val set

  XGBoost (NB03 base)
    AUC-ROC=0.9659  AP=0.7769  F1=0.2642  Recall=0.8228  Precision=0.1574

  Equal Voting (1/3 each)
    AUC-ROC=0.9716  AP=0.7843  F1=0.1702  Recall=0.8734  Precision=0.0943

  Optimal Voting (0.15/0.45/0.40)
    AUC-ROC=0.9719  AP=0.7810  F1=0.3508  Recall=0.8481  Precision=0.2211

  Optimal weights: LR=0.15, RF=0.45, XGB=0.40
